# MCQ Question Generation (QG) -- LoRA Training
**Target:** T4 (15 GB)  |  **Model:** `unsloth/Qwen3-4B-Instruct`

**Setup:** Upload `qg_formatted.jsonl` as a Kaggle Dataset, set `DATA_PATH` in Cell 3, run all.


## 1. GPU Detection


In [ ]:
import os
# ── Pin to single GPU 0 — Unsloth does not support multi-GPU DDP ──────────
# Kaggle exposes 2x T4 but Unsloth can only use one.  Setting this env var
# before any CUDA call ensures the process never sees the second card,
# preventing any accidental DataParallel / device-mismatch errors.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # silence HF tokenizer warning
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Enable GPU T4 x2 in Notebook Settings.")

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
n = torch.cuda.device_count()
print(f"GPU: {name}  |  VRAM: {vram:.1f} GB  |  Count: {n}  |  CUDA: {torch.version.cuda}")

if "T4" not in name:
    raise RuntimeError(
        f"This notebook requires GPU T4 x2. Got: {name}.\n"
        f"Go to Settings → Accelerator → GPU T4 x2"
    )

GPU_TYPE = "T4"
print(f"✅ GPU T4 detected. Proceeding.")


## 2. Install Dependencies


In [ ]:
import subprocess
def _install(cmd, label):
    print(f"[INSTALL] {label} ...", end=" ")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print("OK" if r.returncode == 0 else f"FAIL: {r.stderr[-200:]}")

# Force PyTorch with CUDA 12.1 kernels (supports P100 sm_60 + T4 sm_75)
_install("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q", "PyTorch cu121")
_install("pip install unsloth --upgrade -q", "unsloth")
_install("pip install --upgrade trl transformers peft datasets -q", "trl+transformers+peft+datasets")
_install("pip install rouge-score sentence-transformers scikit-learn structlog numpy -q", "eval deps")

from unsloth import FastLanguageModel
print("\nUnsloth import OK")


## 3. Configuration
Edit `DATA_PATH` to match your uploaded dataset.


In [ ]:
import math
from pathlib import Path

DATA_PATH  = "/kaggle/input/mcq-training-data/qg_formatted.jsonl"
OUTPUT_DIR = "/kaggle/working/mcq_qg"
BASE_MODEL = "unsloth/Qwen3-4B-Instruct"

# Single T4 (15 GB VRAM) -- Unsloth 4-bit.
# Effective batch = 4 x 4 = 16.  Simplified prompts avg ~450 tokens/sample;
# packing fits 2-3 samples per 1024-token window.
BATCH_SIZE     = 4
GRAD_ACCUM     = 4
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT   = True

# QG LoRA: higher rank -- must learn conditioning AND content generation.
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05

EPOCHS = 3
LR     = 2e-4
WD     = 0.01

# T4 does not support bf16; use fp16.
USE_FP16 = True
USE_BF16 = False

# Warmup = 5% of total training steps (computed after data load).
WARMUP_RATIO = 0.05

print(f"Config: batch={BATCH_SIZE} | accum={GRAD_ACCUM} | eff_batch={BATCH_SIZE*GRAD_ACCUM}")
print(f"seq_len={MAX_SEQ_LENGTH} | 4bit={LOAD_IN_4BIT} | r={LORA_R} | alpha={LORA_ALPHA}")
print(f"Model: {BASE_MODEL}")

## 4. Prompt & Parser Helpers


In [ ]:
import re

# Simplified QG training system prompt (~150 tokens).
# Matches what format_qg.py wrote into the 'text' field.
# Used during eval to reconstruct the inference prompt from val sample fields.
_QG_SYSTEM = (
    "You are an expert MCQ generator for a personalized CS learning platform.\n\n"
    "TWO SIGNALS -- TWO DIFFERENT JOBS:\n"
    "  mastery_level controls HOW you frame the question:\n"
    "    vocabulary depth, cognitive register, distractor sophistication\n"
    "  score_category controls HOW HARD the question is:\n"
    "    difficulty level, and whether mastery type is overridden\n\n"
    "Question types:\n"
    "1=Method/API, 2=Code Output, 3=Code Completion,\n"
    "4a=Definition, 4b=Distinction, 4c=Application,\n"
    "4d=Reasoning, 4e=Misconception\n\n"
    "Mastery framing:\n"
    "  Novice: simple vocabulary, what IS or DOES, no cross-concept reasoning\n"
    "  Intermediate: standard CS terms, connect ideas, never pure definition\n"
    "  Expert: precise technical language, WHY it works, tradeoffs\n\n"
    "Score category difficulty:\n"
    "  very_weak: TYPE is always 4a -- ignore mastery for type only,\n"
    "             keep mastery vocabulary; difficulty minimum\n"
    "  weak: one level easier than mastery standard\n"
    "  moderate: standard for mastery\n"
    "  strong: hardest type mastery allows\n\n"
    "OUTPUT FORMAT (required, no exceptions):\n"
    "QUESTION: <question text>\n"
    "ANSWER: <correct answer>\n"
    "EXPLANATION: <why this is correct>\n\n"
    "HARD RULE: When score_category is very_weak -> always Type 4a.\n"
    "HARD RULE: Questions test transferable concepts, never textbook-specific examples.\n"
    "HARD RULE: Never reproduce more than 5 lines of code in a question."
)


def build_qg_eval_prompt(chunk_text, question_type, mastery_level, score_category):
    # Reconstruct the compact user turn used during training, for eval inference.
    user = (
        f"generate question: type: {question_type} | mastery: {mastery_level} | "
        f"score_category: {score_category} | content: {chunk_text}"
    )
    return [{"role": "system", "content": _QG_SYSTEM},
            {"role": "user",   "content": user}]


def apply_chat(messages, tokenizer):
    # Apply chat template with Qwen3 non-thinking mode.
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)


def extract_qg_output(raw):
    if not raw or not raw.strip():
        return None
    t = raw.strip()
    q = re.search(r"QUESTION:\s*(.+?)(?=\nANSWER:|\Z)", t, re.DOTALL)
    a = re.search(r"ANSWER:\s*(.+?)(?=\nEXPLANATION:|\Z)", t, re.DOTALL)
    e = re.search(r"EXPLANATION:\s*(.+)", t, re.DOTALL)
    if q and a:
        qv, av = q.group(1).strip(), a.group(1).strip()
        if qv and av:
            return {"question": qv, "correct_answer": av,
                    "explanation": e.group(1).strip() if e else ""}
    return None


print("Helpers loaded OK")

## 5. Load & Split Data


In [ ]:
import json
import random
from collections import defaultdict
from sklearn.model_selection import train_test_split

def load_jsonl(path):
    samples = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                try: samples.append(json.loads(line))
                except json.JSONDecodeError: pass
    return samples

def stratified_split(samples, val_ratio=0.1):
    # Shuffle before splitting so sequential book order in the source
    # file does not bias the train/val split or batch composition.
    random.seed(42)
    random.shuffle(samples)
    if len(samples) < 10:
        i = max(1, int(len(samples)*(1-val_ratio))); return samples[:i], samples[i:]
    labels = [s.get("question_type","?") for s in samples]
    if len(set(labels)) < 2:
        i = max(1, int(len(samples)*(1-val_ratio))); return samples[:i], samples[i:]
    return train_test_split(samples, test_size=val_ratio, stratify=labels, random_state=42)

all_samples = load_jsonl(DATA_PATH)
print(f"Loaded {len(all_samples)} samples")
td = defaultdict(int)
for s in all_samples: td[s.get("question_type","?")] += 1
print("Type dist:", dict(sorted(td.items())))
train_samples, val_samples = stratified_split(all_samples)
print(f"Train: {len(train_samples)} | Val: {len(val_samples)}")


## 6. Load Model + Apply LoRA


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=LOAD_IN_4BIT, dtype=None,
    device_map={"": 0})
print("Base model loaded.")

model = FastLanguageModel.get_peft_model(model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", use_gradient_checkpointing="unsloth", random_state=42)

tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
tt = sum(p.numel() for p in model.parameters())
print(f"Trainable: {tp:,} / {tt:,} ({100*tp/tt:.3f}%)")


## 7. Build Dataset


In [ ]:
from datasets import Dataset
train_dataset = Dataset.from_list(train_samples)
val_dataset = Dataset.from_list(val_samples)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"Preview: {train_dataset[0]['text'][:200]}")


## 8. Train


In [ ]:
import time
from pathlib import Path
from trl import SFTTrainer, SFTConfig

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Compute warmup steps = 5% of total training steps.
_steps_per_epoch = max(1, len(train_samples) // (BATCH_SIZE * GRAD_ACCUM))
_total_steps     = _steps_per_epoch * EPOCHS
WARMUP           = max(1, round(WARMUP_RATIO * _total_steps))
print(f"steps/epoch~{_steps_per_epoch} | total_steps={_total_steps} | warmup={WARMUP}")

CKPT_DIR = f"{OUTPUT_DIR}/checkpoints"
args = SFTConfig(
    output_dir=CKPT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, warmup_steps=WARMUP, weight_decay=WD, max_grad_norm=1.0,
    fp16=USE_FP16, bf16=USE_BF16,
    # Epoch-based eval/save: one checkpoint per epoch, keeps 2.
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    greater_is_better=False, logging_steps=10, save_total_limit=2,
    report_to="none", seed=42,
    max_seq_length=MAX_SEQ_LENGTH, dataset_text_field="text",
    packing=True, padding_free=False, optim="adamw_8bit",
    dataloader_num_workers=0, dataloader_pin_memory=False,
)

trainer = SFTTrainer(model=model, tokenizer=tokenizer,
    train_dataset=train_dataset, eval_dataset=val_dataset, args=args)

# Resume from the latest checkpoint if one exists.
# This prevents load_best_model_at_end from crashing when the resumed
# epoch loss is worse than a prior checkpoint.
_resume = None
_ckpt_dir = Path(CKPT_DIR)
if _ckpt_dir.exists():
    _ckpts = sorted(
        [d for d in _ckpt_dir.iterdir()
         if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]))
    if _ckpts:
        _resume = str(_ckpts[-1])
        print(f"Resuming from checkpoint: {_resume}")

print(f"Starting QG training | eff_batch={BATCH_SIZE*GRAD_ACCUM} | steps/epoch~{_steps_per_epoch}")
start = time.time()
train_result = trainer.train(resume_from_checkpoint=_resume)
elapsed = round(time.time() - start, 1)
print(f"Done in {elapsed}s | loss={round(train_result.training_loss, 4)}")

## 9. Save LoRA Adapter


In [ ]:
ADAPTER_PATH = f"{OUTPUT_DIR}/adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Saved to: {ADAPTER_PATH}")
for f in sorted(Path(ADAPTER_PATH).iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")


## 10. Evaluate (ROUGE-L + Parse Rate)


In [ ]:
from rouge_score import rouge_scorer as rs_mod
import gc

FastLanguageModel.for_inference(model)
scorer = rs_mod.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
r1, r2, rL = [], [], []
type_rL = defaultdict(list)
ok = fail = 0
N = min(50, len(val_samples))
print(f"Evaluating on {N} samples...")

for s in val_samples[:N]:
    txt = s.get("text", "")

    # Extract reference question/answer from the formatted 'text' field.
    ref_q_m = re.search(r"QUESTION:\s*(.+?)(?=\nANSWER:|$)", txt, re.DOTALL)
    ref_a_m = re.search(r"ANSWER:\s*(.+?)(?=\nEXPLANATION:|$)", txt, re.DOTALL)
    ref_q = ref_q_m.group(1).strip() if ref_q_m else ""
    ref_a = ref_a_m.group(1).strip() if ref_a_m else ""

    # Retrieve conditioning fields stored alongside 'text'.
    qt     = s.get("question_type", "4a")
    ml     = s.get("mastery_level",  "Intermediate")
    sc_cat = s.get("score_category", "moderate")

    # Reconstruct chunk from 'text' -- appears after '| content: ' in the user turn.
    chunk = ""
    cm = re.search(r"\| content:\s*(.+?)(?=\n<\|im_end\|>|\Z)", txt, re.DOTALL)
    if cm:
        chunk = cm.group(1).strip()[:500]

    msgs = build_qg_eval_prompt(chunk, qt, ml, sc_cat)
    inp  = tokenizer(apply_chat(msgs, tokenizer), return_tensors="pt",
                     truncation=True, max_length=MAX_SEQ_LENGTH)
    inp  = {k: v.to(model.device) for k, v in inp.items()}
    ilen = inp["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=200, temperature=0.0, do_sample=False)

    raw = tokenizer.decode(out[0][ilen:], skip_special_tokens=True)
    p   = extract_qg_output(raw)
    if p:
        ok += 1
        if ref_q and ref_a:
            ref = f"QUESTION: {ref_q}\nANSWER: {ref_a}"
            gen = f"QUESTION: {p['question']}\nANSWER: {p['correct_answer']}"
            sc  = scorer.score(ref, gen)
            r1.append(sc["rouge1"].fmeasure)
            r2.append(sc["rouge2"].fmeasure)
            rL.append(sc["rougeL"].fmeasure)
            type_rL[qt].append(sc["rougeL"].fmeasure)
    else:
        fail += 1
    del inp, out
    torch.cuda.empty_cache()

avg = lambda l: round(sum(l) / max(len(l), 1), 4)
print(f"\nROUGE-1={avg(r1)} | ROUGE-2={avg(r2)} | ROUGE-L={avg(rL)} | Parse={ok}/{N}")
for t, v in sorted(type_rL.items()):
    print(f"  Type {t}: ROUGE-L={avg(v)}")
gc.collect()
torch.cuda.empty_cache()

## 11. Save Metrics


In [ ]:
avg = lambda l: round(sum(l) / max(len(l), 1), 4)
metrics = {
    "task": "QG", "gpu": GPU_TYPE, "base_model": BASE_MODEL,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
    "train_loss": round(train_result.training_loss, 4),
    "training_time_s": elapsed,
    "rouge1": avg(r1), "rouge2": avg(r2), "rougeL": avg(rL),
    "parse_ok": ok, "parse_fail": fail,
    "train_n": len(train_samples), "val_n": len(val_samples),
    "warmup_steps": WARMUP, "total_steps": _total_steps,
}
mp = f"{OUTPUT_DIR}/training_metrics.json"
with open(mp, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics: {mp}")
print("Done! Download /kaggle/working/mcq_qg/ from the Output tab.")